# Module 5 — Capstone: Real Multi-Agent System Evaluation

Every evaluation in this tutorial so far has run against a hand-built `LLMTestCase` — a query and an answer you wrote yourself, sometimes with a hand-crafted `retrieval_context` or `tools_called` list attached. This closing capstone runs `TaskCompletionMetric` (Module 2.3) against something else entirely: **the real output of a real, running multi-agent system**, scored in batch, over many trip requests read from a spreadsheet.

_Source: `Agent_Evaluation/CrewAI_Travel_Planner/` — a real CrewAI app (`TripCrew.py`, `tasks.py`, `tools/`, `api_server.py`, `main.py`) plus `evaluate_endtoend_test.py`. Left in place rather than copied here, for the same reason as Module 4's Arize labs: it's a real running multi-process app (a FastAPI backend, a live CrewAI multi-agent run per request), not something that can be flattened into one self-contained notebook cell.

## The app under test: a 3-agent travel planning crew

`TripCrew.py` defines three CrewAI agents, each with a narrow role, run in sequence by a `Crew`:

| Agent | Role | Tools |
|---|---|---|
| `customer_interest_search_agent` | Find places matching the traveler's stated interest in the destination city | `search_internet` (web search via Serper) |
| `local_expert` | Produce a city guide — tourist attractions for the given date range | `search_internet` |
| `travel_concierge` | Assemble everything into a 7-day itinerary with packing and budget suggestions | `search_internet`, `calculate` (arithmetic, for budget math) |

`tasks.py` defines what each agent is asked to produce (`customer_interest_search_tasks`, `trip_info`, `plan_task`), and `TripCrew.run()` wires agents + tasks into a `Crew` and calls `crew.kickoff()` — a sequential multi-agent pipeline where the concierge's final itinerary depends on both earlier agents' outputs. This is a genuinely different shape of system from anything else in this tutorial: not one LLM call, not one agent looping over its own tools (Module 3), but **three separate agents handing off work to each other**.

`api_server.py` wraps `TripCrew` in a small FastAPI app — a single `POST /plan_trip/` endpoint that takes `origin`, `city`, `customer_interest_topic`, `date_range` and returns the finished itinerary. `main.py` is an optional Streamlit front end that calls the same endpoint.

## Standing up the app

This capstone can't run inside this notebook — it needs the FastAPI backend actually running as its own process, with its own dependencies (`crewai`, `langchain-community` for the Serper search tool, `fastapi`, `uvicorn`, none of which are in this tutorial's shared `.venv`) and its own credentials:

```bash
# In Agent_Evaluation/CrewAI_Travel_Planner/, with OPENAI_API_KEY and SERPER_API_KEY set in .env:
pip install crewai langchain-openai langchain-community fastapi uvicorn python-dotenv pandas openpyxl requests pytest deepeval
python api_server.py   # serves POST /plan_trip/ on localhost:8000
```

**A quirk worth knowing about, not silently fixing:** `api_server.py`'s route is `/plan_trip/` (trailing slash) and `main.py`'s Streamlit client calls that exact URL — but `evaluate_endtoend_test.py` (below) posts to `http://localhost:8000/plan_trip` **without** the trailing slash. FastAPI's default redirect behavior means this often still works, but it's the kind of mismatch that's easy to overlook when an eval script and the app it's testing were written at slightly different times — worth checking directly against your FastAPI version rather than assuming, if you actually stand this up.

## The evaluation script

`evaluate_endtoend_test.py` is a `pytest`-parametrized batch test — the same shape as Module 1.6's RAGAS CI pattern, this time scoring with DeepEval's `TaskCompletionMetric` instead. Reproduced here as a walkthrough (not executed — it needs the live backend from above):

```python
from deepeval import evaluate
import pytest
import pandas as pd
import requests
import os
from deepeval.metrics import TaskCompletionMetric
from deepeval.test_case import LLMTestCase, ToolCall
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

EXCEL_FILE = "./trip_questions.xlsx"
BACKEND_URL = "http://localhost:8000/plan_trip"

task_completion = TaskCompletionMetric(threshold=0.7)
test_data = pd.read_excel(EXCEL_FILE)


@pytest.mark.parametrize("row", test_data.iterrows())
def test_trip_planner_with_deepeval(row):
    index, data = row
    origin, city = data["origin"], data["city"]
    interest, date_range = data["interest"], data["date_range"]

    # 1. Call the real, running crew via its API
    payload = {"origin": origin, "city": city,
               "customer_interest_topic": interest, "date_range": date_range}
    response = requests.post(BACKEND_URL, json=payload)
    assert response.status_code == 200, f"API call failed: {response.text}"
    result = response.json()["trip_plan"]

    # 2. Describe the task explicitly -- the real ask is broader than the raw
    #    inputs alone convey, so TaskCompletionMetric's `task` isn't left implicit
    task_description = (
        f"Create a 7-day trip plan from {origin} to {city}, focused on {interest} "
        f"for the date range {date_range}. The output should include: top 3 places "
        f"for the interest, 5 city attractions, a day-by-day itinerary with packing "
        f"and budget tips."
    )

    # 3. Build the test case from REAL output -- not hand-written
    test_case = LLMTestCase(
        input=task_description,
        actual_output=result,
        expected_output="The plan should include detailed travel suggestions "
                        "including points of interest and budget tips.",
        tools_called=[
            ToolCall(name="search internet", input_parameters={"query": task_description}, output=result),
            ToolCall(name="calculate", input_parameters={"operation": "2000 * 2 + 1500"}, output="5500"),
        ],
    )

    task_completion.measure(test_case)
    test_data.at[index, "score"] = task_completion.score
    test_data.at[index, "reason"] = task_completion.reason
    assert 0 <= task_completion.score <= 1

    evaluate(test_cases=[test_case], metrics=[task_completion])


def teardown_module(module):
    test_data.to_excel(EXCEL_FILE, index=False)  # write scores back for review
```

### Reading the script

- **`pytest.mark.parametrize("row", test_data.iterrows())`** turns every row of `trip_questions.xlsx` into its own test — the same "spreadsheet as the batch driver" pattern from Module 1.6's RAGAS example, here parametrizing over trip requests instead of retrieval questions.
- **The real API call is the whole point.** `result = response.json()["trip_plan"]` is genuine output from a live `crew.kickoff()` run — three real agents, real web searches, a real LLM assembling the itinerary — not a string you typed into a test file. This is what Module 0 meant by evaluating "the real, currently-running pipeline end to end" (echoing Module 1.7's capstone, now for an agent system instead of a RAG one).
- **`task_description` is written explicitly, not left to inference.** Recall from Module 2.3: `TaskCompletionMetric`'s `task` parameter is optional and falls back to inferring the goal from `input` — this script chooses to spell it out instead (top-3 places, 5 attractions, day-by-day itinerary, packing/budget tips), because the real requirement is richer than the four raw form fields (`origin`, `city`, `interest`, `date_range`) convey on their own. This is the domain-specific-rubric instinct from Module 2.3's "when 'done' is domain-specific" discussion, applied via the `task` argument instead of a custom `GEval`.
- **The `tools_called` list here is illustrative, not captured from the real crew run.** Notice the values (`"query": task_description`, a fixed `"operation": "2000 * 2 + 1500"`) are hand-set, not pulled from what the CrewAI agents actually did internally — a real integration would need to capture the actual tool calls each agent made (e.g. via CrewAI's own tracing/callbacks) to give `TaskCompletionMetric` a faithful picture of *how* the outcome was reached, not just a plausible-looking one. Worth noticing as a gap between "a batch eval script exists" and "a batch eval script sees everything the metric needs."
- **Scores get written back to the same spreadsheet** (`teardown_module`) — so `trip_questions.xlsx` ends up as both the input dataset and the eval report, one row per trip request, `score`/`reason` columns appended alongside the original inputs.

## What this capstone demonstrates, end to end

This is the same `TaskCompletionMetric` object from Module 2.3 — same import, same `.measure()` call, same score/reason contract — now scoring:

| Module 2.3's example | This capstone |
|---|---|
| One hand-written `LLMTestCase`, run once, printed to the console | Many `LLMTestCase`s, each built from a real API response, run as a `pytest` suite |
| A single hard-coded hotel-booking scenario | Every row of a spreadsheet of trip requests |
| Score printed and discarded | Score written back into the dataset, next to the request that produced it |
| A single LLM's output | The combined output of **three CrewAI agents** handing off work in sequence |

That progression — from "one metric, one hand-built example" to "one metric, batch-scored against a real running multi-agent system, results tracked in a spreadsheet" — is the arc of this entire tutorial in miniature.

## Tutorial complete — where everything landed

| Module | Question it answered |
|---|---|
| 0 | What kinds of evaluation exist, and which tool do you reach for? |
| 1 | Is retrieval good? Is the generated answer grounded, relevant, correct? |
| 2 | Is a conversation coherent? Were the right tools called correctly? Was the task actually completed? |
| 3 | Across a multi-step run, was the *path* efficient and self-correcting, not just the outcome right? |
| 4 | How does all of the above look with real tracing infrastructure and formal experiments? |
| 5 | How does all of the above look against a real, running, multi-agent system? |

Module 0's closing "if you only have an hour" path (0 → 1.2 → 2.2 → 3) covers the four ideas that generalize to almost any system you'll be asked to evaluate. This capstone is the other end of that same arc — the same core ideas, running for real.